In [ ]:
# Step 9 — Ship Mode Performance Analysis

In [1]:
import pandas as pd

df_ship = pd.read_csv(
    r"D:\Nassu Project\Nassau-Candy-Shipping-Analysis\data\processed\feature_engineered_data.csv"
)

df_ship.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,...,Product ID,Product Name,Sales,Units,Gross Profit,Cost,Factory,Shipping Lead Time,Route,Region Route
0,1,US-2021-103800-CHO-MIL-31000,2024-01-03,2026-06-30,Standard Class,103800,United States,Houston,Texas,77095,...,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28,Wicked Choccy's,909,Wicked Choccy's → Texas,Wicked Choccy's → Interior
1,2,US-2021-112326-CHO-TRI-54000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60,Wicked Choccy's,909,Wicked Choccy's → Illinois,Wicked Choccy's → Interior
2,3,US-2021-112326-CHO-NUT-13000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00,Lot's O' Nuts,909,Lot's O' Nuts → Illinois,Lot's O' Nuts → Interior
3,4,US-2021-112326-CHO-SCR-58000,2024-01-04,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30,Lot's O' Nuts,909,Lot's O' Nuts → Illinois,Lot's O' Nuts → Interior
4,5,US-2021-141817-CHO-TRI-54000,2024-01-05,2026-07-05,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,...,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90,Wicked Choccy's,912,Wicked Choccy's → Pennsylvania,Wicked Choccy's → Atlantic


In [ ]:
# Check the shipping modes

print("Ship Mode Distribution:")

display(
    df_ship["Ship Mode"].value_counts()
)

Ship Mode Distribution:


Ship Mode
Standard Class    6120
Second Class      1979
First Class       1548
Same Day           547
Name: count, dtype: int64

In [3]:
# Calculate Ship Mode Performance

# Calculate the same delay benchmark used in previous steps
delay_threshold = df_ship["Shipping Lead Time"].quantile(0.75)

print("Delay Benchmark:", delay_threshold, "days")


# Create delayed shipment indicator
df_ship["Delayed"] = (
    df_ship["Shipping Lead Time"] > delay_threshold
)


# Calculate performance for each ship mode
ship_mode_performance = (
    df_ship.groupby("Ship Mode")
    .agg(
        Shipment_Volume=("Ship Mode", "size"),
        Average_Lead_Time=("Shipping Lead Time", "mean"),
        Minimum_Lead_Time=("Shipping Lead Time", "min"),
        Maximum_Lead_Time=("Shipping Lead Time", "max"),
        Delayed_Shipments=("Delayed", "sum"),
        Delay_Frequency=("Delayed", "mean")
    )
)


# Convert delay frequency into percentage
ship_mode_performance["Delay_Frequency"] = (
    ship_mode_performance["Delay_Frequency"] * 100
)


# Round values for easier reading
ship_mode_performance["Average_Lead_Time"] = (
    ship_mode_performance["Average_Lead_Time"].round(2)
)

ship_mode_performance["Delay_Frequency"] = (
    ship_mode_performance["Delay_Frequency"].round(2)
)


# Sort by average lead time
ship_mode_performance = ship_mode_performance.sort_values(
    "Average_Lead_Time"
)


print("Ship Mode Performance:")
display(ship_mode_performance)

Delay Benchmark: 1638.0 days
Ship Mode Performance:


,Shipment_Volume,Average_Lead_Time,Minimum_Lead_Time,Maximum_Lead_Time,Delayed_Shipments,Delay_Frequency
Ship Mode,,,,,,
Standard Class,6120,1314.33,908,1642,1891,30.90
Second Class,1979,1323.85,906,1640,256,12.94
Same Day,547,1333.44,904,1636,0,0.00
First Class,1548,1338.28,905,1638,0,0.00


In [4]:
# Ship Mode Performance Comparison

ship_mode_comparison = ship_mode_performance[
    [
        "Shipment_Volume",
        "Average_Lead_Time",
        "Delayed_Shipments",
        "Delay_Frequency"
    ]
].copy()

print("Ship Mode Performance Comparison:")

display(ship_mode_comparison)

Ship Mode Performance Comparison:


,Shipment_Volume,Average_Lead_Time,Delayed_Shipments,Delay_Frequency
Ship Mode,,,,
Standard Class,6120,1314.33,1891,30.90
Second Class,1979,1323.85,256,12.94
Same Day,547,1333.44,0,0.00
First Class,1548,1338.28,0,0.00


In [5]:
# Ship Mode Cost-Time Analysis

ship_mode_cost_time = (
    df_ship.groupby("Ship Mode")
    .agg(
        Shipment_Volume=("Ship Mode", "size"),
        Average_Lead_Time=("Shipping Lead Time", "mean"),
        Average_Cost=("Cost", "mean"),
        Average_Sales=("Sales", "mean")
    )
)

ship_mode_cost_time["Average_Lead_Time"] = (
    ship_mode_cost_time["Average_Lead_Time"].round(2)
)

ship_mode_cost_time["Average_Cost"] = (
    ship_mode_cost_time["Average_Cost"].round(2)
)

ship_mode_cost_time["Average_Sales"] = (
    ship_mode_cost_time["Average_Sales"].round(2)
)

print("Ship Mode Cost-Time Analysis:")

display(ship_mode_cost_time)

Ship Mode Cost-Time Analysis:


,Shipment_Volume,Average_Lead_Time,Average_Cost,Average_Sales
Ship Mode,,,,
First Class,1548,1338.28,4.72,13.77
Same Day,547,1333.44,4.41,13.00
Second Class,1979,1323.85,4.83,14.08
Standard Class,6120,1314.33,4.75,13.97


In [6]:
ship_mode_cost_time.to_csv(
    r"D:\Nassu Project\Nassau-Candy-Shipping-Analysis\data\processed\ship_mode_performance_analysis.csv",
    index=True
)

print("Ship mode performance analysis saved successfully.")

Ship mode performance analysis saved successfully.
